In [ ]:
%%capture
!pip install --proxy=192.168.2.10:8080 --upgrade snowflake-connector-python[pandas]
!pip install --proxy=192.168.2.10:8080 --upgrade keyring boto3 pyarrow

In [ ]:
import json
import datetime
from io import BytesIO

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.connector

import warnings
warnings.filterwarnings('ignore')

# Setting Up the Connector

In [ ]:
with open('./credentials.json', 'r') as config_file:
    config = json.load(config_file)

snowflake_creds = config['snowflake_credentials']

ctx = snowflake.connector.connect(**snowflake_creds)
cur = ctx.cursor()

# Load Datasets

In [ ]:
# ── 1. Snowflake — full forecast consolidated table ───────────────────────
cur.execute("""
SELECT * FROM IRONONETECH_DB.PUBLIC.V_FORECAST_CURVE_CONSOLIDATED;
""")
df_sf = cur.fetch_pandas_all()
print(f"Snowflake rows loaded: {len(df_sf):,}")
df_sf.head()

In [ ]:
# ── 2. S3 — Buckeye pre-data (historical, no forecast rows) ──────────────
s3 = boto3.client('s3')

bucket_name = 'fmsp-sagemaker-rstudio-01'
file_key    = 'Forecasting/Buckeye/Forecasting_Buckeye_combined_with_predata.parquet'

obj    = s3.get_object(Bucket=bucket_name, Key=file_key)
df_s3  = pd.read_parquet(BytesIO(obj['Body'].read()))

print(f"S3 rows loaded: {len(df_s3):,}")
df_s3.head()

# Combine Data

Fix the S3 date format to match the Snowflake file, then append the `2025 0+12` forecast rows from Snowflake.

In [ ]:
# ── Fix S3 DATE column → python datetime.date (same format as Snowflake) ─
df_s3['DATE'] = pd.to_datetime(df_s3['DATE']).dt.date

# ── Pull only 2025 0+12 forecast rows from Snowflake for Buckeye ──────────
df_forecast = df_sf[
    (df_sf['PORTFOLIO']     == 'Buckeye') &
    (df_sf['FORECAST_TYPE'] == '2025 0+12')
].copy()
df_forecast['DATE'] = pd.to_datetime(df_forecast['DATE']).dt.date

# ── Keep only the shared columns ──────────────────────────────────────────
COLS = ['FORECAST_TYPE', 'PORTFOLIO', 'SUB_PORTFOLIO', 'METRIC', 'DATE', 'METRIC_VALUE']
df_combined = pd.concat(
    [df_s3[COLS], df_forecast[COLS]],
    ignore_index=True,
)

print(f"Combined rows: {len(df_combined):,}")
print(f"FORECAST_TYPEs : {df_combined['FORECAST_TYPE'].unique().tolist()}")
print(f"SUB_PORTFOLIOs : {df_combined['SUB_PORTFOLIO'].unique().tolist()}")
df_combined.head()

# Filter — Buckeye LP / Ending Outstanding Loans

In [ ]:
df_filtered = df_combined[
    (df_combined['SUB_PORTFOLIO'] == 'LP') &
    (df_combined['METRIC']        == 'Ending Outstanding Loans')
].copy().sort_values('DATE').reset_index(drop=True)

print(f"Filtered rows : {len(df_filtered):,}")
print(f"FORECAST_TYPEs: {df_filtered['FORECAST_TYPE'].unique().tolist()}")
df_filtered

# Visualize Actual Data (Before Fix)

In [ ]:
actual = df_filtered[df_filtered['FORECAST_TYPE'] == 'Actual'].sort_values('DATE')

fig, ax = plt.subplots(figsize=(20, 5))
ax.set_title('Buckeye LP — Ending Outstanding Loans (Actual, before fix)', fontsize=13, fontweight='bold')
ax.plot(actual['DATE'].astype(str), actual['METRIC_VALUE'], marker='o', linewidth=2, color='#2196F3')
ax.set_xlabel('DATE')
ax.set_ylabel('METRIC_VALUE')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Fix 2022-09-30 Zero Value

The `2022-09-30` data point is `0`. Replace it with the average of the preceding month (`2022-08-31`) and the following month (`2022-10-31`).

In [ ]:
target_date = datetime.date(2022, 9, 30)
prev_date   = datetime.date(2022, 8, 31)
next_date   = datetime.date(2022, 10, 31)

actual_idx = (df_filtered['FORECAST_TYPE'] == 'Actual')

val_prev = df_filtered.loc[actual_idx & (df_filtered['DATE'] == prev_date), 'METRIC_VALUE'].values[0]
val_next = df_filtered.loc[actual_idx & (df_filtered['DATE'] == next_date), 'METRIC_VALUE'].values[0]
fill_val = (val_prev + val_next) / 2

df_filtered.loc[
    actual_idx & (df_filtered['DATE'] == target_date),
    'METRIC_VALUE'
] = fill_val

print(f"  2022-08-31 : {val_prev:,.0f}")
print(f"  2022-09-30 : {fill_val:,.0f}  ← filled")
print(f"  2022-10-31 : {val_next:,.0f}")

# Visualize Actual Data (After Fix)

In [ ]:
actual_fixed = df_filtered[df_filtered['FORECAST_TYPE'] == 'Actual'].sort_values('DATE')

fig, ax = plt.subplots(figsize=(20, 5))
ax.set_title('Buckeye LP — Ending Outstanding Loans (Actual, after fix)', fontsize=13, fontweight='bold')
ax.plot(actual_fixed['DATE'].astype(str), actual_fixed['METRIC_VALUE'], marker='o', linewidth=2, color='#2196F3')
ax.set_xlabel('DATE')
ax.set_ylabel('METRIC_VALUE')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Save

In [ ]:
COLS = ['FORECAST_TYPE', 'PORTFOLIO', 'SUB_PORTFOLIO', 'METRIC', 'DATE', 'METRIC_VALUE']
output_path = './buckeye_lp_ending_outstanding_loans.parquet'

df_filtered[COLS].to_parquet(output_path, index=False)
print(f"Saved {len(df_filtered):,} rows → {output_path}")